In [1]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer
from transformers import GenerationConfig
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import torch
print(torch.version.cuda)

c:\Users\User\anaconda3\envs\summarization\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


12.1


In [2]:
train = pd.read_csv('data/train.csv')

In [3]:
validation = train.iloc[700:]
train = train[:700]

In [4]:
train.head()

,paper_id,text,summary
0,0,## FROM SOVEREIGNTY TO EXTRATERRITORIAL CONSCI...,"In this article, Victor Fan argues that analys..."
1,1,## 1. Introduction\n\n\nAn Electronic Health R...,Problem definition: Physicians spend more than...
2,2,## Introduction\n\n\nTranslation plays an i...,Literary translation is one of the most challe...
3,3,## 1 Problem Setup\n\n\nRecent political scien...,There is a long-running debate on evaluating f...
4,4,## INTRODUCTION\n\n\nThis article investigat...,"Recently, ‘bimajyo’ (美魔女) came into focus in J..."


In [5]:
validation.shape

(300, 3)

In [6]:
model_name = 'google/flan-t5-large'
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 558/558 [00:00<00:00, 2823.45it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [7]:
indices = [10]

for i in indices:
    text = train['text'][i]
    summary = train['summary'][i]

    prompt = f"""
Summarize the following text using around 200 words at least {text}

Summary:
"""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    input = tokenizer(prompt, return_tensors='pt')
    output = tokenizer.decode(
        model.generate(input['input_ids'], max_new_tokens=50,)[0]
    )

    print(summary)
    print("--------------------")
    print(output)



[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (5852 > 512). Running this sequence through the model will result in indexing errors


OBJECTIVES: Evidence on how individual characteristics and distancing policies during the first wave of COVID-19 together influenced health behaviours is scarce. The objective of this study is to fill in this gap by studying how the propensity to engage in protective behaviours in Europe was shaped by the interplay of individual characteristics and national policies. 
DESIGN: Data on individual behaviour in 27 countries came from the “Corona Survey” module of the Survey of Health, Ageing and Retirement in Europe, collected in summer 2020. As outcomes, we considered avoidant behaviours (never leaving home, reducing frequency of walks, reducing frequency of social meetings) and preventive behaviour (wearing a face mask). Among relevant policies we considered stay-at-home restrictions, mask wearing policies, and gatherings’ restrictions. Individual characteristics comprised gender, health risk of COVID-19 (older age and poor health), and activity (employment and providing help to other ho

In [8]:
indices = [10, 45, 38]
index = enumerate(indices)

In [9]:
text = train['text'][3]
paragraphs = re.split("\n\n", text)

In [10]:
split = text.split()
[' '.join(split[i:i+500]) for i in range(0, len(split), 500)]

["## 1 Problem Setup Recent political science scholarship has questioned the ability to evaluate presidential election forecasts [1]. On its face, this argument is plausible: presidential elections are rare, and state-level outcomes within presidential elections are highly correlated. If we believe we need 10 or 20 or 100 presidential elections to evaluate whether a forecast provides useful information, we could be waiting a long time wondering if we are being lead astray by forecasts or not. Despite the seeming plausibility of the argument by Grimmer, Knox, and Westwood (henceforth GKW), I argue 1 that there are several ways we may evaluate election forecasts on shorter timescales than 'decades to millennia.' As a heuristic, I argue that we can generally determine whether a forecast is better than random guessing using only publicly available data no more granular than the congressional district level on timescales of two-to-three election cycles (4-6 years in the U.S. counting both m

In [11]:
def preprocessing_function(text):
     # quitar referencias tipo [1]
    text = re.sub(r"\[\d+\]", "", text)
    # limpiar caracteres
    text = re.sub(r"[^a-zA-Z0-9\s.,%\-]", " ", text)
    # normalizar espacios
    text = re.sub(r"\s+", " ", text)

    #cluster_chunker = ClusterSemanticChunker(
    #embedding_function=embedding_function,
    #max_chunk_size=400
    # )
      

In [12]:
def cleaning_text(text):
     # quitar referencias tipo [1]
    text = re.sub(r"\[\d+\]", "", text)
    # limpiar caracteres
    text = re.sub(r"[^a-zA-Z0-9\s.,%\-]", " ", text)
    # normalizar espacios
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [13]:
train['clean_text'] = train['text'].apply(cleaning_text)
validation['clean_text'] = validation['text'].apply(cleaning_text)

In [14]:
def simple_split(text, chunk_size=60):
    txt_split = text.split()
    return [' '.join(txt_split[i:i+chunk_size]) for i in range(0, len(txt_split), chunk_size)]

In [15]:
sentence_transf_model = SentenceTransformer("BAAI/bge-base-en", device='cuda')
def embedding_function(texts):
    return sentence_transf_model.encode(texts, batch_size=64, show_progress_bar=True)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3489.06it/s]


In [16]:
all_chunks = []
chunk_pos = []

for i, text in enumerate(train['clean_text']):
    chunks = simple_split(text)
    all_chunks.extend(chunks)
    chunk_pos.extend([i] * len(chunks))


In [17]:
embeddings = sentence_transf_model.encode(
    all_chunks,
    batch_size=64,
    show_progress_bar=True
)

Batches: 100%|██████████| 1168/1168 [03:34<00:00,  5.46it/s]


In [87]:
def semantic_chunking(chunks, doc_ids, embeddings, window_size=2, t = 0.95, max_len=200):
    #0.9
    merged_chunks = []

    chunks = list(chunks)
    doc_ids = list(doc_ids)
    
    current_chunk = chunks[0]
    current_indices = [0]
    current_doc_id = doc_ids[0]
    
    for i in range(1, len(chunks)):
        
        if doc_ids[i] != current_doc_id:
            merged_chunks.append({
                "text": current_chunk,
                "doc_id": current_doc_id,
                "chunk_indices": current_indices
            })
            
            current_chunk = chunks[i]
            current_indices = [i]
            current_doc_id = doc_ids[i]
            continue
        
        # ventana
        window_indices = current_indices[-window_size:]
        window_embeddings = embeddings[window_indices]
        
        current_embedding = embeddings[i].reshape(1, -1)
        
        sim = cosine_similarity(current_embedding, window_embeddings).max()        
        if sim >= t and len(current_chunk.split()) + len(chunks[i].split()) <= max_len:
            current_chunk += " " + chunks[i]
            current_indices.append(i)
        else:
            merged_chunks.append({
                "text": current_chunk,
                "doc_id": current_doc_id,
                "chunk_indices": current_indices
            })
            
            current_chunk = chunks[i]
            current_indices = [i]
            current_doc_id = doc_ids[i]
    
    # último chunk
    merged_chunks.append({
        "text": current_chunk,
        "doc_id": current_doc_id,
        "chunk_indices": current_indices
    })
    
    return merged_chunks

In [88]:
merged = semantic_chunking(all_chunks, chunk_pos, embeddings)

In [89]:
merged_df = pd.DataFrame(merged)

In [90]:
merged_df_clean = merged_df[merged_df['text'].str.len() > 100]

In [91]:
merged_df_clean.head()

,text,doc_id,chunk_indices
0,FROM SOVEREIGNTY TO EXTRATERRITORIAL CONSCIOUS...,0,[0]
1,of the Euro-American notion of selfdeterminati...,0,[1]
2,"Article 23 of the Hong Kong Basic Law, which r...",0,[2]
3,Chief Executive election in 2017. From Beijing...,0,[3]
4,"Kongers, while Hong Kong is posited within the...",0,[4]


In [92]:
embeddings_chunks = sentence_transf_model.encode(
    merged_df_clean['text'].to_list(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches: 100%|██████████| 1159/1159 [03:38<00:00,  5.29it/s]


In [93]:
embeddings_chunks.shape

(74140, 768)

In [94]:
summary_embedding = sentence_transf_model.encode(
    train['summary'].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

Batches: 100%|██████████| 11/11 [00:05<00:00,  1.88it/s]


In [95]:
summary_per_chunk = summary_embedding[
    merged_df_clean['doc_id'].values
] # Repetir el summary para cada chunk de text creado

In [96]:
merged_df_clean['scores'] = np.sum(
    embeddings_chunks * summary_per_chunk,
    axis=1
) # Al normalizar los vectores, el cosine_similarity se calcula asi de forma equivalente (mirar formula de cosine similarity)

In [97]:
merged_df_clean.head()

,text,doc_id,chunk_indices,scores
0,FROM SOVEREIGNTY TO EXTRATERRITORIAL CONSCIOUS...,0,[0],0.875469
1,of the Euro-American notion of selfdeterminati...,0,[1],0.818374
2,"Article 23 of the Hong Kong Basic Law, which r...",0,[2],0.752425
3,Chief Executive election in 2017. From Beijing...,0,[3],0.765331
4,"Kongers, while Hong Kong is posited within the...",0,[4],0.844045


In [98]:
merged_df_clean['scores'].mean()

np.float32(0.849122)

In [99]:
merged_df_clean['scores'].std()

np.float32(0.043232556)

In [100]:
df_sorted = (
    merged_df_clean
    .sort_values(['doc_id','scores'], ascending=[True, False])
    .groupby('doc_id', group_keys=False)
    .head(6)
    #.apply(lambda x: x.head(max(1, int(len(x)*0.2))))
)

In [101]:
df_sorted

,text,doc_id,chunk_indices,scores
83,or let live. CONCLUSION Drug War offers a pecu...,0,[83],0.901543
17,"argues, colonialism in Hong Kong was character...",0,[17],0.894976
35,"by the Hong Kong viewers, than in renegotiatin...",0,[35],0.891841
16,"as a colonial privilege rather, its affect, th...",0,[16],0.883828
0,FROM SOVEREIGNTY TO EXTRATERRITORIAL CONSCIOUS...,0,[0],0.875469
...,...,...,...,...
74216,perspective can be mobilized to address presen...,699,[74619],0.925293
74291,setting. It is therefore possible that the cur...,699,[74694],0.878064
74293,technologies that power the Great Firewall. Th...,699,[74696],0.874512
74294,"audio to break through restrictions, facilitat...",699,[74697],0.873530


In [102]:
df_sorted['chunk_indices'] = df_sorted['chunk_indices'].apply(lambda x: min(x)) # Desacemos vector de chunk_indices para ordenar el df por esta variable

In [103]:
df_sorted.sort_values(['chunk_indices'], ascending=True, inplace=True)

In [104]:
df_sorted

,text,doc_id,chunk_indices,scores
0,FROM SOVEREIGNTY TO EXTRATERRITORIAL CONSCIOUS...,0,0,0.875469
16,"as a colonial privilege rather, its affect, th...",0,16,0.883828
17,"argues, colonialism in Hong Kong was character...",0,17,0.894976
35,"by the Hong Kong viewers, than in renegotiatin...",0,35,0.891841
83,or let live. CONCLUSION Drug War offers a pecu...,0,83,0.901543
...,...,...,...,...
74216,perspective can be mobilized to address presen...,699,74619,0.925293
74241,With growing tensions between Taiwan and the P...,699,74644,0.872465
74291,setting. It is therefore possible that the cur...,699,74694,0.878064
74293,technologies that power the Great Firewall. Th...,699,74696,0.874512


In [105]:
df_final = df_sorted.groupby('doc_id')['text'].apply(" ".join)

In [106]:
df_final = pd.DataFrame(df_final)

In [107]:
df_final = df_final.join(train['summary'], how='left')

In [108]:
df_final

,text,summary
doc_id,,
0,FROM SOVEREIGNTY TO EXTRATERRITORIAL CONSCIOUS...,"In this article, Victor Fan argues that analys..."
1,"knowledge, recent literature has not incorpora...",Problem definition: Physicians spend more than...
2,strategies that were used to overcome them. Ch...,Literary translation is one of the most challe...
3,1 Problem Setup Recent political science schol...,There is a long-running debate on evaluating f...
4,is the same in Japan. Many Japanese women are ...,"Recently, ‘bimajyo’ (美魔女) came into focus in J..."
...,...,...
695,the fact that the pandemic was in the public s...,Newspapers are a major source of health inform...
696,to explain how the use of collections as the b...,This article provides an account of the making...
697,versus successfully moving passengers effectiv...,Transit in the U.S. is considered secondary to...


In [ ]:
df_final.to_pickle('data/train_clean.pkl')

In [110]:
length = df_final['text'].apply(len)

In [111]:
length

doc_id
0      2364
1      3290
2      2359
3      2262
4      2106
       ... 
695    2356
696    2323
697    2407
698    2537
699    2433
Name: text, Length: 700, dtype: int64

In [112]:
length.mean()

np.float64(2539.42)

In [113]:
length.std()

np.float64(384.5681272814271)

In [114]:
np.percentile(merged_df_clean['scores'], [0, 25, 50, 75, 90, 100])

array([0.67075765, 0.81974298, 0.84875652, 0.87858713, 0.90573923,
       0.98863959])